# megatron/core/tensor_parallel/layers.py：
  - https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/tensor_parallel/layers.py
  - 这是 TP 层的核心定义文件。其中定义了 ColumnParallelLinear、RowParallelLinear 等关键类。
  - 在这些类的 forward 方法中，<font color='red'>实现了计算（矩阵乘法）与通信（All-Gather / Reduce-Scatter）的交错逻辑。</font>
  - 代码中会显式地创建或使用非默认的 CUDA Stream（即“通信流”），将 NCCL 通信操作放入该流中，而将 GEMM（矩阵乘）操作保留在主计算流中，从而实现异步重叠。

## 跨版本兼容

随着 PyTorch 版本的迭代，官方对 AMP 相关 API 的存放位置和参数要求进行了重构。
- <font color='red'>这段代码通过“先尝试新特性，失败则回退到旧写法”的策略，确保同一份代码能在不同版本的 PyTorch 环境中正常运行。</font>

### 1. 核心业务背景
- 在混合精度训练中，开发者经常需要自定义算子（继承 torch.autograd.Function）。
- 为了防止梯度计算时因 FP16 精度不足导致数值不稳定，PyTorch 提供了 <font color='red'>custom_fwd </font>和 <font color='red'>custom_bwd 装饰器</font>，用于控制张量的数据类型转换。

### 第一步： 尝试新版本 API (PyTorch ≥ 2.4)
- 触发条件： 
  - 如果当前环境的 PyTorch 版本大于等于 2.4.0a0。
- API 变更： 
  - <font color='green'>从 PyTorch 2.x 开始，AMP 相关的接口被统一迁移到了 torch.amp 命名空间下。同时，新版 API 强制要求显式指定 device_type 参数（如 "cuda"），以支持更广泛的硬件生态。</font>
- 使用 partial：
  - 因为每次调用都需要带上 device_type="cuda" 比较繁琐，这里使用了 Python 标准库 functools.partial，将参数预先绑定，生成一个无需再传参的新函数对象，方便后续直接作为装饰器使用。
  
### 第二步：回退到旧版本 API (PyTorch < 2.4)

### 第三步：全局异常兜底机制
- 作用： 这是一个防御性编程设计。
  - 如果在执行 is_torch_min_version 检查或导入 torch.amp 时发生任何未预料的错误（例如某些魔改版的 PyTorch、环境依赖冲突等），程序不会崩溃，而是自动降级使用最保守、兼容性最广的旧版 API (torch.cuda.amp)。

In [ ]:
try:
    '''
    如果当前环境的 PyTorch 版本大于等于 2.4.0a0
    '''
    if is_torch_min_version("2.4.0a0"):
        custom_fwd = partial(torch.amp.custom_fwd, device_type="cuda")
        custom_bwd = partial(torch.amp.custom_bwd, device_type="cuda")
    else:
        '''
        回退到旧版本 API (PyTorch < 2.4)
        '''
        custom_fwd = torch.cuda.amp.custom_fwd
        custom_bwd = torch.cuda.amp.custom_bwd
except:
    custom_fwd = torch.cuda.amp.custom_fwd
    custom_bwd = torch.cuda.amp.custom_bwd


## 实现 PyTorch 分布式通信原语的跨版本兼容
- 在分布式训练（如数据并行、模型并行）中，节点间需要频繁进行张量的收集与规约操作。
- 随着 PyTorch 版本的迭代，官方对这些底层通信 API 进行了重构和重命名。这段代码通过“先尝试新特性，失败则回退到旧写法”的策略，确保同一份分布式代码能在不同版本的 PyTorch 环境中正常运行。

### 核心业务背景
- All-Gather（全部聚集）：
  - 将所有进程上的张量收集并拼接成一个完整的张量，且每个进程都能获得这个完整的结果。常用于收集各个 GPU 上的验证集指标或局部输出。
- Reduce-Scatter（规约-散播）：
  - 先将所有进程的数据进行全局规约（如求和），然后将结果按块分割并分发给各个进程。这种操作常用于大 Batch 拆分训练或计算 LayerNorm/BatchNorm 的全局统计量时，以降低单卡显存占用。

### 第一步：尝试新版本 API (PyTorch ≥ 1.13)
- 触发条件： 
  - 如果当前环境的 PyTorch 版本大于等于 1.13.0。
- API 变更： 
  - <font color='red'>从 PyTorch 1.13 开始，官方将这两个底层的高性能集合通信接口正式移入公共命名空间，并更名为 all_gather_into_tensor 和 reduce_scatter_tensor。</font>
  - <font color='green'>它们能直接将结果写入预先分配好的连续内存张量中，避免了中间列表的开销，性能更好。</font>
  
### 第二步：回退到旧版本 API (PyTorch < 1.13)
### 第三步：全局异常兜底机制

In [ ]:
try:
    if is_torch_min_version("1.13.0"):
        dist_all_gather_func = torch.distributed.all_gather_into_tensor
        dist_reduce_scatter_func = torch.distributed.reduce_scatter_tensor
    else:
        dist_all_gather_func = torch.distributed._all_gather_base
        dist_reduce_scatter_func = torch.distributed._reduce_scatter_base
except:
    dist_all_gather_func = torch.distributed._all_gather_base
    dist_reduce_scatter_func = torch.distributed._reduce_scatter_base



## ColumnParallelLinear列并行线性层
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/tensor_parallel/layers.py#L731

### 文档- 具有列并行特性的线性层。

该线性层的计算定义为 Y = XA + b。其中权重矩阵 A 沿其第二维度（即列方向）进行并行化切分，表示为 A = [A_1, ..., A_p]。

- 参数说明:
    - input_size (int): 
        - 矩阵 A 的第一维度大小（输入特征数）。
    - output_size (int): 
        - 矩阵 A 的第二维度大小（输出特征数）。
    - bias (bool): 
        - 如果为 True，则添加偏置项。
    - gather_output (bool): 
        - <font color='red'>如果为 True，将在输出上调用 all-gather 操作，使得完整的输出 Y 对所有 GPU 都可见；</font>
        - 如果为 False，每个 GPU 将仅保留其对应的局部输出 Y_i = XA_i。
    - init_method (callable): 
        - 用于初始化权重的方法。注意：偏置项始终被初始化为零。
    - stride (int): 
        - 用于步幅线性层（strided linear layers）的参数。
    - keep_master_weight_for_test (bool): 
        - 此参数专为测试添加，实际使用时应设置为 False。
        - 若为 True，它将返回用于初始化的主权重（master weights）。
    - skip_bias_add (bool): 
        - 如果为 True，不在本层内部执行加偏置操作，而是将偏置项直接返回给调用者由其自行相加。
        - <font color='red'>这能实现性能优化，因为这样可以将偏置加法与其他逐元素操作（elementwise operations）融合在一起。</font>
    - skip_weight_param_allocation (bool): 
        - 如果为 True，不为权重分配内存参数，必须在前向传播时通过关键字参数 `weight` 传入。
        - 请注意，这不会影响偏置项，如果 bias=True，偏置仍会被正常分配。默认为 False。
    - embedding_activation_buffer (Tensor): 
        - 当启用 defer_embedding_wgrad_compute 功能时，该缓冲区用于在流水线最后阶段保存最终嵌入线性层的输入激活值。
    - grad_output_buffer (Tensor): 
        - 当启用 defer_embedding_wgrad_compute 功能时，该缓冲区用于在流水线最后阶段保存最终嵌入线性层的梯度输出。
    - is_expert (bool): 
        - 如果为 True，则该层将被视为混合专家模型（MoE）中的专家层。
    - config (ModelParallelConfig): 
        - ModelParallelConfig 配置对象。
    - tp_comm_buffer_name (str): 
        - 通信缓冲区的名称，在非 Transformer-Engine 模块中不使用此参数。
    - disable_grad_reduce (bool): 
        - 如果为 True，将禁用跨张量并行组（tensor-parallel ranks）的输出梯度规约（reduction）操作。默认为 False。
        - 该特性主要被 Nemo 框架中的 LoRA Adapter 使用，用于延迟梯度的规约，并将其与其他梯度操作融合以提升性能。

### 初始化

In [ ]:
    def __init__(
        self,
        input_size,
        output_size,
        *,
        config: ModelParallelConfig,
        init_method: Callable,
        bias=True,
        gather_output=False,
        stride=1,
        keep_master_weight_for_test=False,
        skip_bias_add=False,
        skip_weight_param_allocation: bool = False,
        embedding_activation_buffer: Optional[List[torch.Tensor]] = None,
        grad_output_buffer: Optional[List[torch.Tensor]] = None,
        is_expert: bool = False,
        tp_comm_buffer_name: Optional[str] = None,  # Not used
        disable_grad_reduce: bool = False,
        tp_group: Optional[torch.distributed.ProcessGroup] = None,
    ):
        super(ColumnParallelLinear, self).__init__()

        # Keep input parameters
        self.input_size = input_size
        self.output_size = output_size
        self.gather_output = gather_output
        
        # Divide the weight matrix along the last dimension.
        self.skip_bias_add = skip_bias_add
        self.is_expert = is_expert
        self.expert_parallel = config.expert_model_parallel_size > 1
        self.embedding_activation_buffer = embedding_activation_buffer
        self.grad_output_buffer = grad_output_buffer
        self.config = config
        self.disable_grad_reduce = disable_grad_reduce
        self.tp_group = tp_group

        self.tp_group = get_tensor_model_parallel_group_if_none(
            self.tp_group, is_expert=self.is_expert
        )
        world_size = get_pg_size(self.tp_group)
        rank = get_pg_rank(self.tp_group)
        self.explicit_expert_comm = self.is_expert and (world_size > 1 or self.expert_parallel)
        self.output_size_per_partition = divide(output_size, world_size)

        # Parameters.
        # Note: torch.nn.functional.linear performs XA^T + b and as a result
        # we allocate the transpose.
        # Initialize weight.
        '''2-权重（Weight）的初始化策略'''
        
        '''
        跳过分配机制： 
        - 如果 skip_weight_param_allocation=True，则不分配内存（常用于流水线并行或 MoE 专家层的特殊调度）。
        '''
        if not skip_weight_param_allocation:
            '''CPU 初始化逻辑...'''
            '''
            如果在 CPU 上初始化，会先在内存中生成主权重再分发
            '''
            if config.use_cpu_initialization:
                self.weight = Parameter(
                    torch.empty(
                        self.output_size_per_partition, self.input_size, dtype=config.params_dtype
                    )
                )
                if config.perform_initialization:
                    self.master_weight = _initialize_affine_weight_cpu(
                        self.weight,
                        self.output_size,
                        self.input_size,
                        self.output_size_per_partition,
                        0,
                        init_method,
                        stride=stride,
                        return_master_weight=keep_master_weight_for_test,
                        rank=rank,
                        world_size=world_size,
                    )
            else:
                '''GPU 直接初始化逻辑...'''
                '''
                如果在 GPU 上初始化，则直接在显存中分配并调用 _initialize_affine_weight_gpu 进行分布式初始化。
                '''
                self.weight = Parameter(
                    torch.empty(
                        self.output_size_per_partition,
                        self.input_size,
                        device=torch.cuda.current_device(),
                        dtype=config.params_dtype,
                    )
                )
                if config.perform_initialization:
                    _initialize_affine_weight_gpu(
                        self.weight,
                        init_method,
                        partition_dim=0,
                        stride=stride,
                        is_expert=self.is_expert,
                    )

            setattr(self.weight, "allreduce", not (self.is_expert and self.expert_parallel))
        else:
            '''
             如果 skip_weight_param_allocation=True，则不分配内存（常用于流水线并行或 MoE 专家层的特殊调度）
            '''
            self.weight = None

        '''3-偏置项（Bias）的处理'''
        '''
        在列并行中，由于输出被切分了，偏置项也必须随之切分，只保留当前 GPU 对应的那一部分。
        同时，为了避免数值错误，偏置总是被强制初始化为 0。
        这里还通过自定义属性 allreduce 标记了该偏置在反向传播时是否需要跨卡求和。
        '''
        if bias:
            if config.use_cpu_initialization:
                self.bias = Parameter(
                    torch.empty(self.output_size_per_partition, dtype=config.params_dtype)
                )
            else:
                self.bias = Parameter(
                    torch.empty(
                        self.output_size_per_partition,
                        device=torch.cuda.current_device(),
                        dtype=config.params_dtype,
                    )
                )
            set_tensor_model_parallel_attributes(self.bias, True, 0, stride)
            if config.perform_initialization:
                # Always initialize bias to zero.
                '''
                - 分配仅属于当前 GPU 的偏置部分 (output_size_per_partition)
                - 始终将偏置初始化为零
                '''
                with torch.no_grad():
                    self.bias.zero_()
            setattr(self.bias, "allreduce", not (self.is_expert and self.expert_parallel))
        else:
            self.register_parameter("bias", None)

        '''4-序列并行（Sequence Parallel）的互斥校验'''
        
        '''
        序列并行是一种用于减少激活值显存占用的技术，但它必须依赖张量并行。
        如果检测到开启了序列并行但只有单卡（world_size <= 1），代码会自动关闭它并发出警告。
        此外，后面还严格校验了 allreduce_dgrad 和 sequence_parallel 不能同时开启。
        '''
        self.sequence_parallel = config.sequence_parallel
        if self.sequence_parallel and world_size <= 1:
            warnings.warn(
                "`sequence_parallel` is set to `True`, but tensor model parallel size "
                f"is {world_size}. Disabling sequence parallel."
            )
            self.sequence_parallel = False

        '''5-梯度规约（Gradient Reduce）的配置'''
        '''
        在列并行的前向传播结束时，通常需要执行一次 All-Reduce 来聚合各个 GPU 上的结果。
        这个布尔值决定了是否在反向传播时对输入的梯度（dgrad）进行跨卡规约。
        如果使用了 LoRA Adapter 或序列并行，可能会选择延迟或禁用这一操作以融合计算。
        '''
        self.allreduce_dgrad = (
            world_size > 1 and not self.sequence_parallel and not self.disable_grad_reduce
        )

        '''
        6-算子融合（Fusion）的环境检查
        
        梯度累积融合是一项极端的性能优化技术，它依赖于 NVIDIA APEX 库提供的自定义 C++/CUDA 扩展。
        如果用户开启了该特性但环境中没有编译安装对应的底层模块，代码会抛出明确的异常，
        并给出类似 pip install --global-option="--cpp_ext"... 的安装指导，防止程序在训练中途崩溃。
        '''
        if config.gradient_accumulation_fusion and not _grad_accum_fusion_available:
            raise RuntimeError(
                "ColumnParallelLinear was called with gradient_accumulation_fusion set "
                "to True but the custom CUDA extension fused_weight_gradient_mlp_cuda "
                "module is not found. To use gradient_accumulation_fusion you must "
                "install APEX with --cpp_ext and --cuda_ext. For example: "
                'pip install --global-option="--cpp_ext" --global-option="--cuda_ext ." '
                "Note that the extension requires CUDA>=11. Otherwise, you must turn off "
                "gradient accumulation fusion."
            )
        self.gradient_accumulation_fusion = config.gradient_accumulation_fusion

        if self.allreduce_dgrad and self.sequence_parallel:
            raise RuntimeError(
                "`allreduce_dgrad` and `sequence_parallel` cannot be enabled at the same time."
            )

        '''7-状态字典加载的安全钩子'''
        '''
        这是 PyTorch 的一个高级特性。当从旧版 checkpoint 加载模型权重时，
        如果缺少某些新增的状态字段，会导致报错。通过这个预加载钩子，
        可以确保即使旧文件中没有 _extra_state，
        也会自动补上一个空字典，保证了版本升级时的向后兼容性。
        '''
        # Hook adding a default empty _extra_state for state dict
        self._register_load_state_dict_pre_hook(
            lambda state_dict, prefix, *args, **kwargs: state_dict.setdefault(
                f"{prefix}_extra_state"
            )
        )


### forward
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/tensor_parallel/layers.py#L934
- 在这些类的 forward 方法中，<font color='red'>实现了计算（矩阵乘法）与通信（All-Gather / Reduce-Scatter）的交错逻辑。</font>
  - 代码中会显式地创建或使用非默认的 CUDA Stream（即“通信流”），将 NCCL 通信操作放入该流中，而将 GEMM（矩阵乘）操作保留在主计算流中，从而实现异步重叠。
  
- 它的核心作用是:
  - <font color='red'>处理输入张量，执行分布式矩阵乘法，并根据配置决定是否在输出端进行跨 GPU 的通信聚合（All-Gather）。</font>

In [ ]:
from .mappings import (
    copy_to_tensor_model_parallel_region,
    gather_from_sequence_parallel_region,
    gather_from_tensor_model_parallel_region,
    reduce_from_tensor_model_parallel_region,
    reduce_scatter_to_sequence_parallel_region,
    scatter_to_tensor_model_parallel_region,
)

In [ ]:
    def forward(
        self,
        input_: torch.Tensor,
        weight: Optional[torch.Tensor] = None,
        runtime_gather_output: Optional[bool] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Forward of ColumnParallelLinear

        Args:
            input_:
                3D tensor whose order of dimension is [sequence, batch, hidden]
            weight (optional):
                weight tensor to use, compulsory when skip_weight_param_allocation is True.
            runtime_gather_output (bool): Gather output at runtime. Default None means
                `gather_output` arg in the constructor will be used.

        Returns:
            - output
            - bias

        """
        
        '''权重（Weight）的动态获取与校验'''
        if weight is None:
            if self.weight is None:
                raise RuntimeError(
                    "weight was not supplied to ColumnParallelLinear forward pass "
                    "and skip_weight_param_allocation is True."
                )
                
            '''
            如果调用时没有传入 weight，则使用初始化时分配好的 self.weight
            '''    
            weight = self.weight
        else:
            '''
            外部传参模式：
            
            对应构造函数中的 skip_weight_param_allocation=True。
            这种设计常用于流水线并行（Pipeline Parallelism）或 MoE 专家层的特殊调度，权重不常驻显存，
            而是按需从内存加载后作为参数传入。
            同时，代码会严格校验外部传入权重的形状是否与当前 GPU 负责的分片大小一致。
            
            '''
            # Check the weight passed in is the correct shape
            expected_shape = (self.output_size_per_partition, self.input_size)
            if weight.shape != expected_shape:
                raise RuntimeError(
                    f"supplied weight's shape is {tuple(weight.shape)}, "
                    f"not {expected_shape} as expected"
                )

    
        '''偏置项（Bias）的处理策略'''
        '''
        这实现了算子融合的准备工作。
        如果 skip_bias_add=True，前向传播时就不加偏置，
        而是将偏置作为第二个返回值 output_bias 交给调用者。
        这样调用者可以将偏置加法与后续的激活函数（如 GeLU、LayerNorm）融合为一个 CUDA Kernel，
        从而大幅减少显存读写开销。
        '''
        bias = self.bias if not self.skip_bias_add else None
        
        '''
        输入张量的隐式切分（Identity / Split）
        '''
        
        '''
        这是列并行最关键的通信点之一。
        在标准的列并行计算中，每个 GPU 需要完整的输入 X 。如果输入还不是并行的，
        就需要通过 copy_to_tensor_model_parallel_region 将其复制到当前张量并行组的所有设备上。
        但在序列并行（Sequence Parallel）等特定模式下，输入已经是按序列维度切分的，此时不需要额外复制。
        '''
        if (
            self.allreduce_dgrad
            or self.sequence_parallel
            or self.explicit_expert_comm
            or self.disable_grad_reduce
        ):
            input_parallel = input_
        else:
            input_parallel = copy_to_tensor_model_parallel_region(input_, group=self.tp_group)

        '''
        4. 延迟梯度计算的缓冲机制
        
        这是一种极端的性能优化技术。对于网络最后一层的 Embedding 投影，
        反向传播时的权重梯度计算（wgrad）非常耗时且占用大量显存。
        通过开启此选项，前向传播时会把输入激活值缓存到 embedding_activation_buffer 中，
        推迟到后续步骤再统一计算梯度，从而隐藏计算延迟。
        '''
        if self.config.defer_embedding_wgrad_compute:
            if (
                self.config.wgrad_deferral_limit == 0
                or len(self.embedding_activation_buffer) < self.config.wgrad_deferral_limit
            ):
                self.embedding_activation_buffer.append(input_parallel)

        # Matrix multiply.
        allreduce_dgrad = False if self.explicit_expert_comm else self.allreduce_dgrad

        if self.config._cpu_offloading_context is not None:
            if self.config._cpu_offloading_context.inside_context is True:
                if not HAVE_TE:
                    assert (
                        self.config.cpu_offloading is False
                    ), "CPU Offloading cannot be enabled while TE is not present"
                else:
                    input_parallel.activation_offloading = self.config.cpu_offloading_activations

        '''
        核心矩阵乘法（Matrix Multiply）
        
        这里调用了底层的实际计算逻辑 _forward_impl。
        它封装了 PyTorch 原生的 F.linear 以及可能的 APEX 自定义 CUDA 扩展（用于梯度累积融合）。
        
        这一步完成了公式 Yi=X⋅Ai^T + bi的计算，得到局部的输出结果 output_parallel
        '''
        output_parallel = self._forward_impl(
            input=input_parallel,
            weight=weight,
            bias=bias,
            gradient_accumulation_fusion=self.gradient_accumulation_fusion,
            allreduce_dgrad=allreduce_dgrad,
            sequence_parallel=False if self.explicit_expert_comm else self.sequence_parallel,
            grad_output_buffer=(
                self.grad_output_buffer if self.config.defer_embedding_wgrad_compute else None
            ),
            wgrad_deferral_limit=(
                self.config.wgrad_deferral_limit
                if self.config.defer_embedding_wgrad_compute
                else None
            ),
            tp_group=self.tp_group,
        )

        '''
        6. 运行时的动态 All-Gather 决策
        
        静态配置： 默认使用构造函数中设定的 self.gather_output。
        动态覆盖： 允许在运行时通过 runtime_gather_output 参数临时改变行为。
                  这在推理阶段或某些特殊的 Transformer 架构中非常有用。
        通信执行： - 如果需要收集，就调用 gather_from_tensor_model_parallel_region 执行 All-Gather 操作，
                   将所有 GPU 上的局部输出 Yi拼接成完整的 Y ；
                  - 如果不需要收集（例如紧接着下一个层就是行并行层），则直接返回局部结果，省去一次昂贵的跨卡通信。
        '''
        gather_output = self.gather_output
        # Use the runtime gather output if it's set explicitly.
        if runtime_gather_output is not None:
            gather_output = runtime_gather_output

        if gather_output:
            # All-gather across the partitions.
            output = gather_from_tensor_model_parallel_region(output_parallel, group=self.tp_group)
        else:
            output = output_parallel
        output_bias = self.bias if self.skip_bias_add else None
        return output, output_bias


## LinearWithGradAccumulationAndAsyncCommunication
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/tensor_parallel/layers.py#L433
LinearWithGradAccumulationAndAsyncCommunication 是 Megatron-LM 框架中用于大模型分布式训练的一个核心自定义算子。
- 它继承自 torch.autograd.Function，
- <font color='red'>主要被封装在 linear_with_grad_accumulation_and_async_allreduce 函数中，</font><font color='blue'>供列并行（ColumnParallelLinear）和行并行（RowParallelLinear）的线性层调用。</font>

- 这个类的核心使命是在反向传播阶段实现两大极致优化：
  - <font color='red'>梯度累积融合（Gradient Accumulation Fusion）</font>
  - <font color='blue'>与异步通信重叠（Async Communication Overlap）。</font>

以下是其内部逻辑的详细拆解：

### 1. 前向传播（Forward Pass）

在前向传播阶段，该算子主要负责准备输入数据并执行矩阵乘法：
- 状态保存： 使用上下文对象 ctx 保存输入张量（input）和权重（weight），以及偏置、序列并行等配置标志，<font color='red'>以便在反向传播时恢复。</font>
- 序列并行（Sequence Parallel）支持： 
  - 如果启用了序列并行，由于每个 GPU 只持有一部分序列的数据，前向传播会先通过 _all_gather_base 操作从所有参与并行的 GPU 上收集完整的输入数据（total_input）。
  - 如果不启用序列并行，则直接使用原始输入。
- 矩阵计算： 对准备好的输入数据和权重的转置进行矩阵乘法（GEMM），并在需要时加上偏置，返回输出结果。

### 2. 反向传播（Backward Pass）—— 核心优化所在

<font color='red'>反向传播是这个类发挥性能优势的关键，它将梯度计算与跨卡通信进行了深度解耦和重叠：</font>

#### (1) 异步 All-Gather 与 dgrad 计算重叠
- 在反向传播初期，为了计算权重的梯度（wgrad），通常需要完整的输入数据。此时，代码会异步发起一个 _all_gather_base 通信操作（async_op=True），让 NCCL 在后台搬运数据。
- 与此同时，主计算流不会等待，而是立即利用上游传来的梯度（grad_output）和权重，开始计算关于输入的梯度（dgrad = grad_output @ weight）。
- 当后续真正需要完整输入来计算 wgrad 时，再通过 handle.wait() 阻塞等待通信完成。

#### (2) 异步 All-Reduce / Reduce-Scatter 与 wgrad 计算重叠
当 dgrad 计算完成后，通常需要将其聚合到各个设备上。为了避免这步通信阻塞后续的 wgrad 计算，代码会根据是否开启序列并行选择不同的异步通信策略：
- 常规模式： 启动异步的 all_reduce 操作来聚合输入梯度。
- 序列并行模式： 启动异步的 _reduce_scatter_base 操作，<font color='red'>将聚合后的梯度分片发回给各个 GPU。</font>

<b>在这两个异步通信进行的同时，主计算流可以无缝衔接去计算权重的梯度（wgrad），从而实现了完美的通信与计算重叠。</b>


#### (3) 梯度累积融合（Gradient Accumulation Fusion）
- 在传统的 PyTorch 训练中，计算完 wgrad 后，还需要单独执行一次加法操作（weight.grad += new_wgrad），这会带来额外的全局显存读写开销。

- 该算子通过引入自定义 CUDA 扩展模块（如 fused_weight_gradient_mlp_cuda），<font color='red'>将“矩阵乘法计算梯度”与“累加到现有梯度缓冲区”这两步合并为一个底层 Kernel。</font>无论是 FP32 还是 FP16/BF16 精度，都可以直接在 GEMM 接口中将当前结果累加到 weight.main_grad 上。
- <font color='red'>这不仅省去了额外的加法算子，还大幅提升了显存带宽利用率。</font>

### 💡 总结与工程约束

LinearWithGradAccumulationAndAsyncCommunication 完美体现了 Megatron-LM 榨干硬件性能的工程设计。
- 需要注意的是，为了让这种细粒度的任务调度生效，使用该模块时必须设置环境变量 CUDA_DEVICE_MAX_CONNECTIONS=1。<font color='red'>这一设置能强制 GPU 按照代码调用的顺序执行内核，确保通信操作被正确地安排在计算操作之前</font>，从而实现最大程度的加速效果。

## linear_with_grad_accumulation_and_async_allreduce
- 它的主要职责是作为外部调用的统一入口，在执行核心的自定义算子之前，进行环境变量的安全检查和参数打包。

### 在反向传播中支持异步通信与梯度累积融合的线性层执行算子。

- 该模块提供了将反向传播的计算结果直接累加到现有梯度缓冲区中的选项，从而避免了在梯度计算完成后还需要额外执行一次加法算子（addition kernel）。

- 此外，输入梯度的张量并行 All-Reduce 操作可以与权重梯度的计算异步并行执行。

- 在序列并行（Sequence Parallelism）模式下，输入梯度的 Reduce-Scatter 操作
同样可以与权重梯度的计算异步并行执行。

### 【重要环境要求】
- 使用此模块必须设置环境变量：CUDA_DEVICE_MAX_CONNECTIONS=1。
- 代码中有一些集合通信操作（collective operations）需要在计算内核（compute kernels）之前被调度， 以便实现通信与计算的重叠。这对于提升性能（加速）是必要的，但对计算的正确性并非必需，
- 因此调度器本身并不会强制保证这种执行顺序。
  - 设置 CUDA_DEVICE_MAX_CONNECTIONS=1 会强制内核严格按照它们被调用的顺序进行调度，从而确保重叠生效。

### 参数说明:
   - input (torch.Tensor, 必需): 
        - 输入张量，用法同 torch.nn.functional.linear。
   - weight (torch.Tensor, 必需): 
        - 权重张量，用法同 torch.nn.functional.linear。
   - bias (torch.Tensor, 可选): 
        - 偏置张量，用法同 torch.nn.functional.linear。
   - gradient_accumulation_fusion (bool, 必需): 
        - 是否执行梯度累积融合。<font color='red'>这需要自定义的 CUDA 扩展模块 fused_weight_gradient_mlp_cuda。</font>
        - <font color='red'>要使用该特性，必须安装带有 --cpp_ext 和 --cuda_ext 选项的 APEX 库。</font>
        - 例如："pip install --global-option=\"--cpp_ext\" --global-option=\"--cuda_ext .\""
        - 注意：该扩展要求 CUDA 版本 >= 11。否则，必须关闭梯度累积融合功能。
   - allreduce_dgrad (bool, 必需): 
        - 是否对输入梯度执行 All-Reduce 操作。
        - 该 All-Reduce 操作会与权重梯度的计算异步进行。
        - 如果 sequence_parallel 为 True，则此参数必须为 False，因为序列并行模式下不会执行 All-Reduce。
   - sequence_parallel (bool, 必需): 
        - <font color='red'>指示是否使用了序列并行。</font>
        - 如果为 True，在前向传播阶段会对输入执行 All-Gather 操作，
          - 在反向传播阶段会对输入梯度执行 Reduce-Scatter 操作。
   - tp_group (torch.distributed.ProcessGroup, 必需): 
        - 用于张量并行操作的进程组。
   - grad_output_buffer (List[torch.Tensor], 可选): 
        - 当嵌入表（embedding table）的权重梯度（wgrad）计算被延迟时，用于保存输出梯度的缓冲区。
        - 默认为 None。
   - wgrad_deferral_limit (int, 可选): 
        - 限制需要延迟执行嵌入层权重梯度 GEMM 计算的微批次（micro-batches）数量。
        - 设置为 0 即可禁用此功能。默认为 0。

In [ ]:
def linear_with_grad_accumulation_and_async_allreduce(
    input: torch.Tensor,
    weight: torch.Tensor,
    bias: Optional[torch.Tensor],
    gradient_accumulation_fusion: bool,
    allreduce_dgrad: bool,
    sequence_parallel: bool,
    grad_output_buffer: Optional[List[torch.Tensor]] = None,
    wgrad_deferral_limit: Optional[int] = 0,
    tp_group: Optional[torch.distributed.ProcessGroup] = None,
) -> torch.Tensor:

    '''
    
    如果调用时没有显式传入张量并行的进程组（tp_group），
    则通过 get_tensor_model_parallel_group_if_none 自动获取默认的张量并行组，
    确保后续的分布式通信能够正确路由。

    '''
    tp_group = get_tensor_model_parallel_group_if_none(tp_group)

    '''
    将所有的输入参数打包成一个列表。
    这是因为底层真正执行计算的是继承自 torch.autograd.Function 的自定义类，
    其 .apply() 方法需要以位置参数的形式接收这些变量。
    '''
    args = [
        input,
        weight,
        bias,
        gradient_accumulation_fusion,
        allreduce_dgrad,
        sequence_parallel,
        grad_output_buffer,
        wgrad_deferral_limit,
        tp_group,
    ]

    
    '''
    这是该函数非常关键的一个工程防御机制。正如之前文档提到的，要实现通信与计算的重叠，
    必须设置 CUDA_DEVICE_MAX_CONNECTIONS=1 来强制内核按顺序调度。
    '''
    if not linear_with_grad_accumulation_and_async_allreduce.warned:
        if os.environ.get("CUDA_DEVICE_MAX_CONNECTIONS") != "1":
            if sequence_parallel:
                '''
                如果检测到环境变量未设置为 "1"，
                且开启了 sequence_parallel 或 allreduce_dgrad，
                函数会抛出警告，提示用户当前的配置无法达到最大加速效果。
                '''
                warnings.warn(
                    "When using sequence parallelism it is recommended to set the "
                    "environment variable CUDA_DEVICE_MAX_CONNECTIONS to 1 for "
                    "maximum speedup"
                )
                '''
                防重复机制
                通过这种“函数级静态变量”的设计，警告只会在程序运行初期触发一次，随后将标志位设为 True，保证了日志的整洁。
                '''
                linear_with_grad_accumulation_and_async_allreduce.warned = True

            if allreduce_dgrad:
                warnings.warn(
                    "When using async grad allreduce it is recommended to set the "
                    "environment variable CUDA_DEVICE_MAX_CONNECTIONS to 1 for "
                    "maximum speedup"
                )
                linear_with_grad_accumulation_and_async_allreduce.warned = True

    '''
    在完成环境检查和参数准备后，
    s函数将解包后的参数传递给 LinearWithGradAccumulationAndAsyncCommunication 类的 apply 方法。
    这一步正式将控制权交给了我们在之前讨论过的自定义 Autograd Function，
    由其接管前向传播和反向传播中的异步 All-Gather/All-Reduce 以及梯度累积融合等底层 CUDA 操作。
    '''
    
    return LinearWithGradAccumulationAndAsyncCommunication.apply(*args)


linear_with_grad_accumulation_and_async_allreduce.warned = False


In [ ]:
class LinearWithGradAccumulationAndAsyncCommunication(torch.autograd.Function):

### apply

- 详见 train_code/VERL/Megatron-LM/torch/torch.autograd.Function/1-apply.ipynb

### forward
- 它负责执行带有张量并行（Tensor Parallelism）和序列并行（Sequence Parallelism）支持的前向矩阵乘法。

#### 装饰器与上下文状态保存
- @custom_fwd：
  - 这是 Apex 库提供的装饰器，主要用于处理混合精度训练（Mixed Precision Training）时的数据类型转换，确保前向传播在正确的数据类型（如 FP16/BF16）下执行。
  
####  序列并行（Sequence Parallelism）的前向 All-Gather

- 序列并行的核心思想：
  - 在序列并行模式下，输入张量（通常是激活值）是被切分（Split）到不同的 GPU 上的。但在进行矩阵乘法时，需要完整的输入。
- 维度计算：
  - 获取当前输入的维度，并将第一个维度（通常是序列长度或 Batch Size）乘以 GPU 的数量（tp_group.size()），得到 All-Gather 后完整张量的目标形状。
- 内存复用：
  - 通过 get_global_memory_buffer() 获取一个全局预分配的内存缓冲区。这避免了每次前向传播都动态分配和释放显存的开销。
- All-Gather 通信：
  - 调用分布式通信原语，将所有 GPU 上的输入分片拼接成完整的 total_input。如果没有开启序列并行，则直接使用原始输入。
  
#### 总结
这个 forward 方法完美体现了 Megatron-LM 的工程设计：
- 通信与计算的结合：
  - 在矩阵乘法前，无缝嵌入了序列并行所需的 All-Gather 通信。
- 显存优化：
  - 通过全局内存缓冲区和巧妙的 ctx 状态保存策略，避免了不必要的显存分配和 PyTorch 的安全检查开销。
- 为反向传播铺路：
  - 提前准备好了 main_grad 的引用和各种控制标志，使得后续的 backward 方法能够高效地执行异步通信和梯度累积融合。



In [ ]:
    """See linear_with_grad_accumulation_and_async_allreduce"""

    @staticmethod
    @custom_fwd
    def forward(
        ctx,
        input,
        weight,
        bias,
        gradient_accumulation_fusion,
        allreduce_dgrad,
        sequence_parallel,
        grad_output_buffer,
        wgrad_deferral_limit,
        tp_group,
    ):
        
        """Forward."""
        if gradient_accumulation_fusion and hasattr(weight, "main_grad"):
            '''
            梯度累积融合准备：
            - 如果开启了梯度累积融合（gradient_accumulation_fusion），
            并且权重张量上挂载了 main_grad 属性（这是 Megatron-LM 用于存储累积梯度的缓冲区），
            则将其提取出来。
            '''
            main_grad = weight.main_grad
        else:
            main_grad = None
            
        '''
        上下文保存策略：
          input 和 weight 是反向传播计算梯度所必需的，通过 ctx.save_for_backward 保存。
          PyTorch 会对这些张量进行特殊的内存管理（例如在不需要时释放）。
         
         关键细节：注释中特别解释了为什么 main_grad 不能放入 save_for_backward。
                  因为这个线性层模块可能会在多个层之间复用（例如 MTP 多 Token 预测的 logits 层）。
                  如果放入 save_for_backward，可能会触发 PyTorch 的就地修改（in-place modification）检查报错。
                  因此，直接将其作为普通属性保存在 ctx 中。
        '''
        ctx.save_for_backward(input, weight)
        # We can't save main_grad in save_for_backward as this module would be
        # reused across layers like MTP logits. So, to prevent in-place modification
        # checks we save the tensor in ctx.
        ctx.main_grad = main_grad
        
        '''
        保存控制标志：将 use_bias, allreduce_dgrad, 
        sequence_parallel 等布尔标志和进程组 tp_group 保存到 ctx，以便在 backward 阶段使用
        '''
        ctx.use_bias = bias is not None
        ctx.gradient_accumulation_fusion = gradient_accumulation_fusion
        ctx.allreduce_dgrad = allreduce_dgrad
        ctx.sequence_parallel = sequence_parallel
        ctx.wgrad_deferral_limit = wgrad_deferral_limit
        ctx.grad_output_buffer = grad_output_buffer
        ctx.tp_group = tp_group

        if sequence_parallel:
            dim_size = list(input.size())
            dim_size[0] = dim_size[0] * tp_group.size()
            '''
            内存复用：通过 get_global_memory_buffer() 获取一个全局预分配的内存缓冲区。
                     这避免了每次前向传播都动态分配和释放显存的开销
            '''
            all_gather_buffer = get_global_memory_buffer().get_tensor(dim_size, input.dtype, "mpu")
            '''
            All-Gather 通信：调用分布式通信原语，将所有 GPU 上的输入分片拼接成完整的 total_input。
                            如果没有开启序列并行，则直接使用原始输入。
            '''
            dist_all_gather_func(all_gather_buffer, input, group=tp_group)
            total_input = all_gather_buffer
        else:
            total_input = input

        '''
        使用完整的输入 total_input 和权重的转置 weight.t() 进行矩阵乘法（即 Y=X⋅W^T）
        '''
        output = torch.matmul(total_input, weight.t())
        if bias is not None:
            output = output + bias
        return output


### backward

- 它实现了大模型分布式训练中极其复杂的反向传播逻辑，
- 核心目标是通过通信与计算重叠（Overlap）以及算子融合（Fusion）来最大化 GPU 的利用率。

#### 通信计算重叠：
- 在序列并行下，计算权重梯度需要完整的输入。代码异步发起 All-Gather 通信，然后立即开始计算输入梯度（grad_input = grad_output.matmul(weight)）。这样，网络通信和矩阵乘法在 GPU 上并行执行。

#### 单连接保证：
- 注释中反复强调依赖 CUDA_DEVICE_MAX_CONNECTIONS=1。如果不加此限制，GPU 硬件调度器可能会乱序执行，导致在通信完成前就开始计算，从而引发数据错误。

#### 梯度累积融合：
- 如果开启了该优化，Wgrad 的计算结果会直接累加到 weight.main_grad 中，而不是先生成一个临时张量再相加。这省去了大量的显存读写（HBM 带宽）。

#### 防死锁机制：
- 在 FSDP 等分布式框架中，如果梯度直接在后台线程更新，可能会阻塞反向传播钩子。因此代码会生成一个 dummy（占位）的 grad_weight 返回给 PyTorch 引擎，确保主线程的 Autograd 钩子能正常触发。

#### 常规回退：
- 如果没有开启融合，则使用标准的矩阵乘法计算 grad_weight

In [ ]:
    @staticmethod
    @custom_bwd
    def backward(ctx, grad_output):
        
        '''上下文恢复'''
        """Backward."""
        input, weight = ctx.saved_tensors
        
        main_grad = ctx.main_grad
        use_bias = ctx.use_bias
        grad_output_buffer = ctx.grad_output_buffer
        wgrad_deferral_limit = ctx.wgrad_deferral_limit
        handle = None
        tp_group = ctx.tp_group

        if ctx.gradient_accumulation_fusion:
            weight.main_grad = main_grad

        '''
        Wgrad 延迟计算：
          - 这是一个针对 Embedding 层等特定场景的高级优化。
            如果配置了 grad_output_buffer 且未达到延迟上限，代码会将 grad_output 存入缓冲区，
            并将 wgrad_compute 设为 False。
            这意味着当前层暂时不计算权重梯度，而是推迟到后续统一处理，以优化显存和计算调度。
        '''
        wgrad_compute = True
        if grad_output_buffer is not None:
            if wgrad_deferral_limit == 0 or len(grad_output_buffer) < wgrad_deferral_limit:
                grad_output_buffer.append(grad_output)
                wgrad_compute = False

        '''
        异步 All-Gather 与输入梯度（Dgrad）计算的重叠
        '''
        if wgrad_compute:
            if ctx.sequence_parallel:
                dim_size = list(input.size())
                dim_size[0] = dim_size[0] * tp_group.size()

                all_gather_buffer = get_global_memory_buffer().get_tensor(
                    dim_size, input.dtype, "mpu"
                )
                '''异步发起 All-Gather 通信'''
                handle = dist_all_gather_func(
                    all_gather_buffer, input, group=tp_group, async_op=True
                )

                '''
                注释强调：依赖 CUDA_DEVICE_MAX_CONNECTIONS=1 保证 通信 在 计算前 被调度
                '''
                # Here we rely on CUDA_DEVICE_MAX_CONNECTIONS=1 to ensure that the
                # gather is scheduled before the input gradient computation
                total_input = all_gather_buffer
            else:
                total_input = input
        '''
        计算输入梯度 (Dgrad)
        '''
        grad_input = grad_output.matmul(weight)

        if ctx.sequence_parallel and wgrad_compute:
            '''等待 All-Gather 完成'''
            # pylint: disable=possibly-used-before-assignment
            handle.wait()

        if wgrad_compute:
            grad_output, total_input = prepare_input_tensors_for_wgrad_compute(
                grad_output, total_input
            )

        '''
        输入梯度的异步聚合（All-Reduce 或 Reduce-Scatter）
        '''
        if ctx.allreduce_dgrad:
            
            '''
            Dgrad 聚合：计算出 grad_input 后，需要跨 GPU 聚合。
                       在常规张量并行中，使用 all_reduce。
                       在序列并行中，使用 reduce_scatter（将梯度规约并切分，正好对应前向传播的 All-Gather 逆操作）。
            '''
            # Asynchronous all-reduce
            '''异步 All-Reduce'''
            handle = torch.distributed.all_reduce(grad_input, group=tp_group, async_op=True)
            # Here we rely on CUDA_DEVICE_MAX_CONNECTIONS=1 to ensure that the
            # all-reduce is scheduled before the weight gradient computation

        if ctx.sequence_parallel:
            assert not ctx.allreduce_dgrad
            dim_size = list(input.size())
            sub_grad_input = torch.empty(
                dim_size, dtype=input.dtype, device=torch.cuda.current_device(), requires_grad=False
            )
            '''异步 Reduce-Scatter'''
            '''
            再次重叠：这个聚合操作同样是异步的，它将与下一步的权重梯度（Wgrad）计算重叠。
            '''
            # reduce_scatter
            handle = dist_reduce_scatter_func(
                sub_grad_input, grad_input, group=tp_group, async_op=True
            )
            # Here we rely on CUDA_DEVICE_MAX_CONNECTIONS=1 to ensure that the
            # reduce scatter is scheduled before the weight gradient computation

        if ctx.gradient_accumulation_fusion:
            if wgrad_compute:
                '''
                根据数据类型调用不同的 CUDA 融合算子
                '''
                # In case of Megatron-FSDP, need to create main grad buffers in-place
                if hasattr(weight, "__fsdp_param__"):
                    weight.main_grad = weight.get_main_grad()
                    torch.matmul(grad_output.t(), total_input, out=weight.main_grad)
                else:
                    if weight.main_grad.dtype == torch.float32:
                        fused_weight_gradient_mlp_cuda.wgrad_gemm_accum_fp32(
                            total_input, grad_output, weight.main_grad
                        )
                    elif weight.main_grad.dtype in (torch.float16, torch.bfloat16):
                        fused_weight_gradient_mlp_cuda.wgrad_gemm_accum_fp16(
                            total_input, grad_output, weight.main_grad
                        )
                    else:
                        raise RuntimeError(
                            "Unsupported gradient type for gradient accumulation fusion"
                        )

            '''
            处理 FSDP 和 Megatron-FSDP 的 dummy grad 逻辑，防止后台线程死锁
            '''
            if hasattr(weight, "grad_added_to_main_grad"):
                # When overlap_grad_reduce is True, need to ensure that backward hooks
                # are all run on the main backprop thread to prevent deadlocks. Setup
                # dummy grad_weight tensor to prevent backward hooks from being run
                # in a background thread.
                if getattr(weight, "zero_out_wgrad", False):
                    if HAVE_TE:
                        # get_dummy_wgrad function in TE enables reuse of single dummy wgrad buffer
                        # across different layers/microbatches. The function accepts shape as list.
                        grad_weight = get_dummy_wgrad(
                            list(weight.main_grad.shape), input.dtype, zero=True
                        )
                    else:
                        grad_weight = torch.zeros(
                            weight.main_grad.shape,
                            dtype=input.dtype,
                            device=torch.cuda.current_device(),
                            requires_grad=False,
                        )
                else:
                    if HAVE_TE:
                        grad_weight = get_dummy_wgrad(list(weight.main_grad.shape), input.dtype)
                    else:
                        grad_weight = torch.empty(
                            weight.main_grad.shape,
                            dtype=input.dtype,
                            device=torch.cuda.current_device(),
                            requires_grad=False,
                        )
                weight.grad_added_to_main_grad = True
            else:
                grad_weight = None
        else:
            '''常规 Wgrad 计算'''
            '''
            同步点：
                 在返回梯度之前，必须 handle.wait() 确保所有的异步通信都已经完成，否则返回的梯度数据是不完整的。
            '''
            grad_weight = grad_output.t().matmul(total_input)
            
        grad_bias = grad_output.sum(dim=0) if use_bias else None

        if ctx.sequence_parallel:
            '''
            等待 Reduce-Scatter 完成
            '''
            handle.wait()
            # Need to return None's as gradient has to flow for all the input arguments
            # provided during forward
            return (sub_grad_input, grad_weight, grad_bias, None, None, None, None, None, None)

        if ctx.allreduce_dgrad:
            handle.wait()

        '''
        返回值对齐：
        backward 返回的梯度数量必须与 forward 的输入参数数量严格一致。
        由于 forward 接收了 9 个参数，这里返回了 9 个值，其中非 Tensor 参数（如布尔标志）对应的梯度位置填充为 None
        '''
        return grad_input, grad_weight, grad_bias, None, None, None, None, None, None
